# Volatility Regression 02 — five-minute feature provenance audit

**Goal:** determine whether missing L2/option predictors reflect (1) absent raw collector fields, (2) historical collector-vintage NULLs, (3) raw→canonical→market→final transformation loss, (4) legitimate quality/quote restrictions, or (5) the all-predictor OLS complete-case policy.

**This is a read-only forensic audit, not a new volatility forecast.** No data are collected, no model is refitted, and no upstream tables, gates, or collector settings are changed. v0.2.0: **all available expiries** are inspected. This branch is independent of HSMM, so HSMM 06's confirmation window is not a constraint here (Ivan, 2026-10-09). The section reproducing Vol Reg 01's complete-case loss keeps 01's own cutoff (2026-10-07) so it still describes 01. The raw source is the existing five-minute Parquet aggregate, not tick data.

**Important distinction:** a raw field present in the unioned Parquet schema can still be absent from older daily file schemas. Both are reported. Any error opening local raw files is a diagnostic failure, **not** evidence of missing raw values.

**Interpretation rule:** schema absence, native nulls, and quality-gate masking are separate. A feature excluded from OLS due to coverage is not a zero-predictive-power finding.

Reusable implementation: `src/data_engineering/modeling/feature_provenance_audit.py`. Outputs: `02_*.csv` and `02_run_metadata.json`.

## 1. Reproducible read-only inputs and synthetic tests

In [ ]:
from pathlib import Path
from datetime import timedelta
import os, sys, json, subprocess
import numpy as np
import pandas as pd
import duckdb
from dotenv import load_dotenv

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "src" / "data_engineering").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Launch from repository root or a child directory")
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")
DATA_ROOT = os.getenv("DERIBIT_DATA_ROOT")
if not DATA_ROOT:
    raise RuntimeError("DERIBIT_DATA_ROOT missing in project .env")
DB = Path(DATA_ROOT) / "database" / "research.duckdb"
if not DB.is_file():
    raise FileNotFoundError(DB)
FUTURES = Path(DATA_ROOT) / "futures" / "5m"
OUTPUT = ROOT / "notebooks" / "Volatility_Regression"
END = None              # no cutoff in this branch (Ivan, 2026-10-09)
END_01 = "2026-10-07"   # Vol Reg 01 reproduction only
from data_engineering.modeling import feature_provenance_audit as pa
from data_engineering.modeling import vol_ols_5m as ols
print("Provenance version", pa.FEATURE_PROVENANCE_VERSION)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s",
                str(ROOT / "tests"), "-p", "test_feature_provenance_audit.py", "-v"],
               cwd=ROOT, check=True)
print("Synthetic provenance checks completed.")

## 2. Audit the existing five-minute raw Parquet and its collector vintages

In [ ]:
def sqlstr(v):
    return "'" + str(v).replace("'", "''") + "'"

def select_available(con, relation_sql, requested):
    cols = set(con.execute(f"DESCRIBE SELECT * FROM {relation_sql}").df()["column_name"])
    keep = [c for c in requested if c in cols]
    if "underlying" not in cols:
        raise ValueError("Relation missing underlying key")
    return keep, cols

with duckdb.connect(str(DB), read_only=True) as con:
    con.execute("SET TimeZone = 'UTC'")
    where = "" if END is None else "WHERE TRY_CAST(expiry_date AS DATE) <= CAST(? AS DATE)"
    final = con.execute(f"""
        SELECT * FROM time_series_feature_state
        {where}
        ORDER BY underlying, state_time
    """, [] if END is None else [END]).df()
if final.empty:
    raise ValueError("No feature states found")
final = pa.normalize_stage(final)
lo = final.state_time.min().date() - timedelta(days=1)
hi = final.state_time.max().date()
files = sorted(p for p in FUTURES.glob("date=*/futures_5m.parquet")
               if lo <= pd.Timestamp(p.parent.name.split("=",1)[1]).date() <= hi)
if not files:
    raise FileNotFoundError(f"No futures/5m date-partitioned files for {lo}..{hi}")
raw_relation = "read_parquet([" + ", ".join(sqlstr(p.as_posix()) for p in files) +                "], union_by_name=true, hive_partitioning=true)"
raw_fields = ["underlying", "bar_end", *pa.L2_COLUMNS, *pa.EXTRA_RAW_COLUMNS,
              "book_twa_spread", "trade_count"]
schema_rows = []
with duckdb.connect(str(DB), read_only=True) as con:
    con.execute("SET TimeZone = 'UTC'")
    for file in files:
        single = f"read_parquet({sqlstr(file.as_posix())})"
        colnames = set(con.execute(f"DESCRIBE SELECT * FROM {single}").df()["column_name"])
        for name in (*pa.L2_COLUMNS, *pa.EXTRA_RAW_COLUMNS):
            schema_rows.append({"date": file.parent.name, "file": str(file.relative_to(FUTURES)),
                                "field": name, "field_in_file_schema": name in colnames,
                                "file_size_bytes": file.stat().st_size})
    raw_present, raw_schema = select_available(con, raw_relation, raw_fields)
    if "bar_end" not in raw_schema:
        raise ValueError("Raw futures Parquet has no bar_end")
    raw = con.execute(f"""
        SELECT {', '.join(raw_present)}
        FROM {raw_relation}
        WHERE bar_end BETWEEN ? AND ?
        ORDER BY underlying, bar_end
    """, [final.state_time.min().to_pydatetime(),
          final.state_time.max().to_pydatetime()]).df()
raw = pa.normalize_stage(raw, time_col="bar_end")
schema_df = pd.DataFrame(schema_rows)
print(f"Feature states {len(final):,}; raw rows {len(raw):,}; raw files {len(files)}")
display(schema_df.groupby("field")["field_in_file_schema"].agg(["sum","count"]))

## 3. Align raw → canonical → engineered market → final model states

In [ ]:
stage_columns = list(dict.fromkeys(["underlying", "state_time", *pa.L2_COLUMNS,
                 *pa.EXTRA_RAW_COLUMNS, *pa.GATES,
                 *pa.OPTION_CHANGES, *pa.OPTION_LEVELS]))
upstream = {}
schema_by_stage = [{"stage": "raw", "field": x, "schema_present": x in raw_schema}
                   for x in (*pa.L2_COLUMNS, *pa.EXTRA_RAW_COLUMNS)]
with duckdb.connect(str(DB), read_only=True) as con:
    con.execute("SET TimeZone = 'UTC'")
    for name, relation in (("canonical", "canonical_market_state"),
                           ("market", "market_time_series_state")):
        selected, cols = select_available(con, relation, stage_columns)
        if "state_time" not in cols:
            raise ValueError(f"{relation} missing state_time")
        rows = con.execute(f"""
            SELECT {', '.join(selected)} FROM {relation}
            WHERE state_time BETWEEN ? AND ?
            ORDER BY underlying, state_time
        """, [final.state_time.min().to_pydatetime(),
              final.state_time.max().to_pydatetime()]).df()
        upstream[name] = rows
        schema_by_stage.extend({"stage": name, "field": x,
                                "schema_present": x in cols}
                               for x in (*pa.L2_COLUMNS, *pa.EXTRA_RAW_COLUMNS))
stages, alignment_df = pa.align_stages(final, {
    "raw": raw, "canonical": upstream["canonical"], "market": upstream["market"]
})
# Dataframe columns are only projected when they exist, avoiding silent NULL
# injection that would obscure genuine schema absence.
stage_df = pa.stage_coverage(stages, fields=pa.L2_COLUMNS)
cause_df = pa.l2_attribution(stage_df)
quality_df = pa.gate_summary(stages["final"], stages["raw"])
stage_schema_df = pd.DataFrame(schema_by_stage)
display(alignment_df)
display(cause_df.groupby(["field", "diagnosis"]).size().unstack(fill_value=0))
if (alignment_df["duplicate_key_rows"] > 0).any():
    print("WARNING: duplicate upstream keys observed; see 02_alignment.csv.")

## 4. Collector versions, L2 gate funnel and midpath aggregate availability

In [ ]:
r = stages["raw"].copy()
r["expiry_date"] = stages["final"]["expiry_date"].astype(str).to_numpy()
for col in ("collector_version", "book_obi_20_std", "mid_path_realized_variance",
            "mid_path_valid_points", "book_stats_complete"):
    if col not in r:
        r[col] = np.nan
r["collector_version"] = r["collector_version"].fillna("<unknown>")
version_df = r.groupby(["underlying","expiry_date","collector_version"],
                       dropna=False).agg(
    states=("state_time","size"),
    l2_std_nonnull=("book_obi_20_std", "count"),
    midpath_rv_nonnull=("mid_path_realized_variance", "count"),
    midpath_points_nonnull=("mid_path_valid_points", "count")
).reset_index()
display(version_df.groupby(["collector_version"], dropna=False)[
    ["states","l2_std_nonnull","midpath_rv_nonnull"]].sum())
display(quality_df.groupby("underlying")[
    ["n_model_states","raw_obi_std_nonnull","raw_book_stats_complete",
     "l2_intrawindow_ready","perp_integrity_ok",
     "instability_features_ready"]].sum())
print("Raw non-null counts do not imply that a value passed the research gate.")

## 5. Options: quote/settlement masks and 1h change history

In [ ]:
options_df = pa.option_hour_coverage(stages["final"])
display(options_df[["underlying","utc_hour","states",
                    "in_settlement_window_true","option_snapshot_ready_true",
                    "option_atm_quote_ok_true","atm_iv_change_1h_native"]]
        .query("utc_hour in [7,8,9]").sort_values(["underlying","utc_hour"]))
print("Expected at 08:00: session-local 60m differences require 12 previous 5m states.")
print("Observe quote/source eligibility separately from actual raw collection.")

## 5b. Added checks (v0.2.0, VR 007)

1. **Unmatched states** by expiry and UTC hour, with each asset's first raw bar. Expected: the 09-20 and 09-21 sessions and the first hours of 09-22 predate the collector.
2. **NULL index returns across assets.** If NULLs coincide across all assets they are collector-wide gaps (restarts/outages), not asset-specific staleness. `realized_vol_4h` needs 48/48 returns, so each gap removes 4 h of it.
3. **Collector vintages derived from the data** (version stamp → event-time audit field → L2 instability fields), and the **standardized mean shift of every model feature within ±24 h of each vintage boundary**. A shift confined to L2/collector-derived features, with index/trade features stable, indicates a definitional break.


In [ ]:
from data_engineering.modeling.full_information import FEATURE_SPECS

unmatched_df = pa.unmatched_states(final, raw)
by_expiry = (unmatched_df.groupby("expiry_date")[["states", "unmatched"]].sum()
             .query("unmatched > 0"))
print("Unmatched model states by expiry (all assets):")
display(by_expiry)

null_by_time_df, null_summary_df = pa.null_return_simultaneity(final)
print("NULL 5m index returns: isolated vs shared across assets")
display(null_summary_df.round(3))

raw_aligned = stages["raw"]
vintage = pa.collector_vintage(raw_aligned)
boundaries_df = pa.vintage_boundaries(vintage, stages["final"]["state_time"],
                                      stages["final"]["underlying"])
display(boundaries_df)

families = {s.name: s.block for s in FEATURE_SPECS}
shift_frames = []
for asset in sorted(final.underlying.unique()):
    pos = np.flatnonzero(stages["final"]["underlying"].to_numpy() == asset)
    g = stages["final"].iloc[pos]
    order = np.argsort(g["state_time"].to_numpy(), kind="stable")
    meta, X, _ = ols.make_panel(g)          # sorted by state_time
    v = vintage.iloc[pos].iloc[order].reset_index(drop=True)
    sh = pa.boundary_shift(X, v, meta["state_time"], families=families)
    sh.insert(0, "underlying", asset)
    shift_frames.append(sh)
shift_df = pd.concat(shift_frames, ignore_index=True) if shift_frames else pd.DataFrame()
if not shift_df.empty:
    s = shift_df.assign(abs_shift=shift_df["std_mean_shift"].abs())
    print("Median |standardized shift| by boundary and feature family:")
    display(s.groupby(["boundary", "family"])["abs_shift"].median().unstack().round(2))


## 6. Precisely measure complete-case loss in Vol Reg 01 (availability only)

In [ ]:
waterfalls, missingness, failures = [], [], []
for asset in sorted(final.underlying.unique()):
    g = final.loc[final.underlying == asset].copy()
    try:
        w, m = pa.model_coverage(g, end=END_01)
        waterfalls.append(w)
        missingness.append(m)
    except Exception as e:
        failures.append({"underlying": asset, "reason": f"{type(e).__name__}: {e}"})
waterfall_df = pd.concat(waterfalls, ignore_index=True) if waterfalls else pd.DataFrame()
feature_missing_df = pd.concat(missingness, ignore_index=True) if missingness else pd.DataFrame()
failures_df = pd.DataFrame(failures)
if not waterfall_df.empty:
    train = waterfall_df[waterfall_df["split"] == "train"].copy()
    train["share_complete"] = train.n_complete / train.n_eligible.replace(0, np.nan)
    display(train.pivot(index="underlying", columns="spec", values="share_complete").round(3))
if not failures_df.empty:
    display(failures_df)
print("Exclusion variants are NOT regressions. They only measure retained rows.")
print("No held-out volatility scores have been consulted for feature selection.")

## 7. Export evidence, not assumptions

In [ ]:
outputs = {
    "02_raw_file_schema.csv": schema_df,
    "02_stage_schema.csv": stage_schema_df,
    "02_alignment.csv": alignment_df,
    "02_stage_coverage.csv": stage_df,
    "02_l2_attribution.csv": cause_df,
    "02_quality_gates.csv": quality_df,
    "02_collector_versions.csv": version_df,
    "02_option_hour_coverage.csv": options_df,
    "02_complete_case_waterfall.csv": waterfall_df,
    "02_feature_missingness.csv": feature_missing_df,
    "02_errors.csv": failures_df,
    "02_unmatched_states.csv": unmatched_df,
    "02_null_return_by_time.csv": null_by_time_df,
    "02_null_return_summary.csv": null_summary_df,
    "02_vintage_boundaries.csv": boundaries_df,
    "02_vintage_boundary_shift.csv": shift_df,
}
for name, frame in outputs.items():
    frame.to_csv(OUTPUT / name, index=False)
    print(name, "rows:", len(frame))
meta = {
    "version": pa.FEATURE_PROVENANCE_VERSION,
    "audit_type": "read_only_raw_canonical_market_final_model",
    "expiry_cutoff": END,
    "vol_reg_01_reproduction_cutoff": END_01,
    "features_checked": list(pa.L2_COLUMNS),
    "futures_files_inspected": len(files),
    "model_state_rows": len(final),
    "raw_state_rows": len(raw),
    "no_database_writes": True,
    "branch_independent_of_hsmm_holdout": True,
    "model_fits_changed": False,
    "notes": [
        "Collector file schema and raw value NULLs are different diagnoses",
        "Readiness flags overlap; do not add missing counts as disjoint causes",
        "Complete-case removal is a model availability diagnostic, not predictive evidence",
        "Results require manual review before proposing pipeline changes"
    ],
    "model_coverage_errors": failures,
}
(OUTPUT / "02_run_metadata.json").write_text(
    json.dumps(meta, indent=2), encoding="utf-8")
print("Audit export complete — commit executed notebook and 02_* outputs.")

## Interpretation and safety contract

- Check `02_raw_file_schema.csv` first. If old daily files never had the L2 field, downstream NULLs are **collector vintage**, not a bug.
- If a field exists in raw data but disappears in the canonical/market layer, investigate the stage and the optional-column SQL; do not automatically alter gates.
- If raw data survive but `instability_features_ready` is false, inspect `book_stats_complete`, `l2_intrawindow_ready` and `perp_integrity_ok` jointly. These counts overlap and cannot be summed into exclusive failure reasons.
- `02_option_hour_coverage.csv` distinguishes native option-change NULLs from quote and settlement gates; option changes requiring lagged states are expected to be absent early in sessions.
- `02_complete_case_waterfall.csv` compares availability when option or L2 predictors are excluded **without refitting**. A larger available sample is not automatically better forecasting performance.
- The 10-second midprice-path fields are inventoried **only as already-collected five-minute aggregates**. No target substitution or tick storage.
- Vintage boundary shifts compare ±24 h windows; market moves also shift features, so read L2 shifts **relative to** index/trade features at the same boundary.